In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "50")

In [0]:
df = spark.read.csv("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv", header=True, inferSchema=True)
display(df)

In [0]:
# Filter high-risk employees: still employed but low job satisfaction
high_risk_df = df.filter((df.Attrition == "No") & (df.JobSatisfaction < 3))

# Select relevant informative columns
high_risk_df = high_risk_df.select(
    "EmployeeNumber", "Age", "Gender", "Department", "JobRole", "JobLevel",
    "JobSatisfaction", "EnvironmentSatisfaction", "MonthlyIncome",
    "YearsAtCompany", "OverTime", "BusinessTravel", "DistanceFromHome",
    "PerformanceRating", "Attrition"
)

# Write to Delta table in external-catalog.default
high_risk_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.`default`.`high_risk_employees`")

display(high_risk_df)

In [0]:
display(spark.sql("DESCRIBE HISTORY `external-catalog`.`default`.`high_risk_employees`"))

In [0]:
df_delta = spark.read.table("`external-catalog`.`default`.`high_risk_employees`")
display(df_delta)

In [0]:
spark.sql("""
    INSERT INTO `external-catalog`.`default`.`high_risk_employees`
    VALUES (9999, 35, 'Male', 'Sales', 'Sales Executive', 2, 1, 2, 5000, 3, 'No', 'Travel_Rarely', 5, 3, 'No')
""")

In [0]:
display(spark.sql("DESCRIBE HISTORY `external-catalog`.`default`.`high_risk_employees`"))

In [0]:
spark.read.format('delta').load('/path/to/delta/table')
display (df_delta)

In [0]:
df_version = spark.read.format("delta").option("versionAsOf", 0).table("`external-catalog`.`default`.`high_risk_employees`")
display(df_version)

In [0]:
df_timestamp = spark.read.format("delta").option("timestampAsOf", "2026-10-04T06:58:25.667Z").table("`external-catalog`.`default`.`high_risk_employees`")
display(df_timestamp)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catalog`.`default`.`employee_transformed_data`")

In [0]:
from pyspark.sql.functions import when, col, round as spark_round

# Apply logical transformations on the employee dataframe
transformed_df = df \
    .withColumn("AnnualIncome", spark_round(col("MonthlyIncome") * 12, 2)) \
    .withColumn("TenureGroup",
        when(col("YearsAtCompany") < 2, "New")
        .when(col("YearsAtCompany") < 5, "Junior")
        .when(col("YearsAtCompany") < 10, "Mid")
        .otherwise("Senior")) \
    .withColumn("AvgSatisfaction",
        spark_round((col("EnvironmentSatisfaction") + col("JobSatisfaction") + col("RelationshipSatisfaction")) / 3, 2))

# Write to volume partitioned by Department as Parquet
transformed_df.write.format("parquet") \
    .mode("overwrite") \
    .partitionBy("Department") \
    .save("/Volumes/external-catalog/default/employee_transformed_data/employee_data_parquet")

display(transformed_df)